# Learning an Insertion-Sort Invariant with RoboVerify

This tutorial starts with a concrete insertion-sort input, records outer-loop snapshots, converts them into relational truth rows, and asks RoboVerify to infer a universally quantified candidate invariant.

The final result is **candidate invariant inference**, not a proof of the Python program. Formal induction would additionally require symbolic array-update semantics and verification conditions.

## 1. Imports and project setup

The notebook imports the reusable relational front end and the insertion-sort example from the project source tree.

In [1]:
from pathlib import Path
import inspect
import sys

import z3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from synthesis.examples.insertion_sort import (
    Pair,
    Solution,
    build_insertion_sort_domain,
    build_training_snapshots,
    check_sorted_prefix_entailment,
    pairs_from_keys,
    snapshots_have_sorted_prefix,
    sorted_prefix_target,
    to_relational_snapshot,
    trace_insertion_sort,
)
from synthesis.inference_lib.relational import (
    build_relational_dataset,
    build_relational_vocabulary,
    ground_relational_row,
    infer_relational_invariants,
)

## 2. The supplied insertion-sort algorithm

The source version fixes the constructor spelling to `__init__` while preserving the supplied backward scan, slice shifting, and stable `<=` comparison.

In [2]:
print(inspect.getsource(Pair))
print(inspect.getsource(Solution))

class Pair:
    """A sortable key and its associated value."""

    def __init__(self, key: int, value: str):
        self.key = key
        self.value = value

    def __repr__(self) -> str:
        return f"Pair(key={self.key}, value={self.value!r})"

class Solution:
    """The supplied stable, slice-shifting insertion-sort implementation."""

    def insertionSort(self, pairs: List[Pair]) -> List[List[Pair]]:
        list_of_lists: List[List[Pair]] = []

        if pairs is None or pairs == []:
            return list_of_lists

        list_of_lists.append(pairs.copy())

        for i in range(1, len(pairs)):
            pair = pairs[i]
            for j in range(i):
                idx = i - j - 1
                if pairs[idx].key <= pair.key:
                    pairs[idx + 2 : i + 1] = pairs[idx + 1 : i]
                    pairs[idx + 1] = pair
                    break
                if idx == 0:
                    pairs[1 : i + 1] = pairs[0:i]
                    pairs[0] = 

## 3. Run one concrete input

We begin with the original three pairs. The algorithm sorts by `key`; `value` lets us track each object's identity.

In [3]:
walkthrough_pairs = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]
walkthrough_run = trace_insertion_sort(walkthrough_pairs)
[(pair.key, pair.value) for pair in walkthrough_run.sorted_pairs]

[(2, 'banana'), (5, 'apple'), (9, 'cherry')]

## 4. Record one consistent program point

A loop invariant must hold at the same control-flow location. We record the initial outer-loop head, every later outer-loop head, and the loop exit. `outer_index` is also the processed-prefix length.

In [4]:
def labeled_order(snapshot):
    return [
        f"{name}:{snapshot.keys[name]}:{snapshot.values[name]}"
        for name in snapshot.order
    ]

for snapshot in walkthrough_run.snapshots:
    print(
        f"i={snapshot.outer_index}",
        f"order={labeled_order(snapshot)}",
        f"processed={sorted(snapshot.processed)}",
    )

i=1 order=['item_0:5:apple', 'item_1:2:banana', 'item_2:9:cherry'] processed=['item_0']
i=2 order=['item_1:2:banana', 'item_0:5:apple', 'item_2:9:cherry'] processed=['item_0', 'item_1']
i=3 order=['item_1:2:banana', 'item_0:5:apple', 'item_2:9:cherry'] processed=['item_0', 'item_1', 'item_2']


## 5. Stable objects versus changing positions

`item_0`, `item_1`, and `item_2` name the original `Pair` objects. Their keys and values never change. Sorting changes only their order. This distinction lets relational predicates refer to the same object across snapshots.

In [5]:
walkthrough_snapshots = tuple(
    to_relational_snapshot(snapshot)
    for snapshot in walkthrough_run.snapshots
)
walkthrough_snapshots[1]

RelationalSnapshot(objects=('item_0', 'item_1', 'item_2'), payload=InsertionSortSnapshot(outer_index=2, order=('item_1', 'item_0', 'item_2'), processed=frozenset({'item_1', 'item_0'}), keys={'item_0': 5, 'item_1': 2, 'item_2': 9}, values={'item_0': 'apple', 'item_1': 'banana', 'item_2': 'cherry'}), constant_bindings={})

## 6. Define the relational vocabulary

The insertion-sort domain supplies three predicates:

- `Processed(x)`: object `x` belongs to the current processed prefix.
- `Before(x, y)`: `x` occurs before `y` in the current array.
- `KeyLE(x, y)`: `x.key <= y.key`.

`Before` is state-dependent, `Processed` changes as the prefix grows, and `KeyLE` is fixed by object keys. Equality is added by the generic adapter.

In [6]:
domain = build_insertion_sort_domain()
second_state = walkthrough_snapshots[1]
{
    "Processed(item_1)": domain.evaluate("Processed", second_state, ("item_1",)),
    "Before(item_1, item_0)": domain.evaluate("Before", second_state, ("item_1", "item_0")),
    "KeyLE(item_1, item_0)": domain.evaluate("KeyLE", second_state, ("item_1", "item_0")),
}

{'Processed(item_1)': True,
 'Before(item_1, item_0)': True,
 'KeyLE(item_1, item_0)': True}

## 7. Choose the quantifier bound and construct atoms

Sortedness compares two objects, so we choose `k = 2`. RoboVerify creates `ux1` and `ux2`, then applies every allowed predicate to those symbolic terms.

In [7]:
k = 2
universal_variables, vocabulary = build_relational_vocabulary(domain, k)
print("Universal variables:", universal_variables)
for index, atom in enumerate(vocabulary):
    print(f"{index}: {atom}")

Universal variables: (ux1, ux2)
0: Processed(ux1)
1: Processed(ux2)
2: Before(ux1, ux2)
3: Before(ux2, ux1)
4: KeyLE(ux1, ux2)
5: KeyLE(ux2, ux1)
6: ux1 == ux2


## 8. Ground one assignment into a Boolean row

In the second snapshot, assign `ux1 = item_1` (banana, key 2) and `ux2 = item_0` (apple, key 5). Each vocabulary atom becomes `True` or `False`.

In [8]:
example_row = ground_relational_row(
    domain,
    second_state,
    vocabulary,
    universal_variables,
    assignment=("item_1", "item_0"),
)
for atom, value in zip(vocabulary, example_row):
    print(f"{str(atom):24s} -> {value}")

Processed(ux1)           -> True
Processed(ux2)           -> True
Before(ux1, ux2)         -> True
Before(ux2, ux1)         -> False
KeyLE(ux1, ux2)          -> True
KeyLE(ux2, ux1)          -> False
ux1 == ux2               -> False


## 9. Build the dataset

For every snapshot, RoboVerify enumerates every ordered assignment of the two universal variables to concrete objects. Duplicate Boolean rows are collapsed.

In [9]:
walkthrough_rows = build_relational_dataset(
    domain, walkthrough_snapshots, vocabulary, universal_variables
)
print("Walkthrough snapshots:", len(walkthrough_snapshots))
print("Distinct walkthrough truth rows:", len(walkthrough_rows))

Walkthrough snapshots: 3
Distinct walkthrough truth rows: 10


## 10. Add diverse traces before learning

One execution can support accidental correlations. The training corpus therefore includes sorted, reverse, middle-insertion, smallest-first, duplicate-key, all-equal, and different-length cases.

In [10]:
training_snapshots = build_training_snapshots()
training_rows = build_relational_dataset(
    domain, training_snapshots, vocabulary, universal_variables
)
print("Training snapshots:", len(training_snapshots))
print("Distinct training truth rows:", len(training_rows))
print("Every training prefix is concretely sorted:", snapshots_have_sorted_prefix(training_snapshots))

Training snapshots: 30
Distinct training truth rows: 18
Every training prefix is concretely sorted: True


## 11. Infer universally quantified clauses

The back end treats each atom as a target, finds a minimum set of distinguishing predicates, synthesizes Boolean clauses, universally quantifies them, and removes clauses implied by domain axioms or earlier retained clauses.

In [11]:
result = infer_relational_invariants(
    domain, training_snapshots, k=2, verbose=False
)
print("Vocabulary size:", len(result.vocabulary))
print("Truth-row count:", len(result.truth_rows))
print("Retained learned clauses:", len(result.clauses))

Vocabulary size: 7
Truth-row count: 18
Retained learned clauses: 2


In [12]:
for index, clause in enumerate(result.clauses, start=1):
    print(f"Clause {index}")
    print("  expression:", clause.expr)
    print("  target atom:", clause.target_predicate)
    print("  learned via:", clause.learned_via)

Clause 1
  expression: ForAll([ux1, ux2],
       Or(Not(Processed(ux2)),
          Before(ux2, ux1),
          Processed(ux1)))
  target atom: Processed(ux1)
  learned via: phi
Clause 2
  expression: ForAll([ux1, ux2],
       Or(Not(Processed(ux1)),
          Before(ux1, ux2),
          KeyLE(ux2, ux1)))
  target atom: Processed(ux1)
  learned via: phi_prime


## 12. Interpret the learned result

The retained conjunction expresses two ideas: processed membership is prefix-closed, and objects at or before a processed object have no larger key. Together with the strict total-order axioms for `Before`, these imply the familiar sorted-prefix invariant.

The target property is:

$$
\forall x,y.\; Processed(x) \land Processed(y) \land Before(x,y)
\Rightarrow KeyLE(x,y).
$$

In [13]:
target = sorted_prefix_target(domain)
print(target)

ForAll([target_x, target_y],
       Implies(And(Processed(target_x),
                   Processed(target_y),
                   Before(target_x, target_y)),
               KeyLE(target_x, target_y)))


## 13. Check semantic entailment with Z3

We ask whether the domain axioms, the learned invariant, and the negation of the target can all hold. `unsat` means no countermodel exists, so the learned conjunction entails the target under the stated axioms.

In [14]:
entailment_result = check_sorted_prefix_entailment(domain, result.invariant)
print("Axioms and learned invariant entail the target:", entailment_result)
assert entailment_result == z3.unsat

Axioms and learned invariant entail the target: unsat


## 14. Validate held-out executions

These inputs were not used to construct the training corpus. Concrete validation is useful evidence against overfitting, but it is still not a proof over every possible execution.

In [15]:
held_out_snapshots = []
for case_index, keys in enumerate(((7, 3, 7, 1), (-1, 4, 0), (8,))):
    held_out_run = trace_insertion_sort(
        pairs_from_keys(keys, f"held_out_{case_index}")
    )
    held_out_snapshots.extend(
        to_relational_snapshot(snapshot)
        for snapshot in held_out_run.snapshots
    )
print("Held-out snapshots:", len(held_out_snapshots))
print("Every held-out processed prefix is sorted:", snapshots_have_sorted_prefix(held_out_snapshots))

Held-out snapshots: 8
Every held-out processed prefix is sorted: True


## What has and has not been established

**Established in this tutorial**

1. The Python algorithm produced outer-loop snapshots.
2. Those snapshots were translated into the declared relational vocabulary.
3. RoboVerify learned quantified clauses from the resulting Boolean rows.
4. Under the domain axioms, the learned conjunction entails the intended sorted-prefix formula.
5. The target also held on several held-out traces.

**Not established**

- The learned formula has not been proved inductive for arbitrary arrays.
- Permutation preservation is not represented by this vocabulary.
- Stability is tested for the implementation but is not part of the learned formula.
- The inner-loop invariant is outside this tutorial.

A full proof would need a symbolic array model, integer index semantics, weakest-precondition rules for shifts and assignments, and verification conditions for initialization, maintenance, and termination.